In [1]:
print('hello world')

hello world


# AI Agents Hands-On Lab

Today you're going to build a real agent — not a slide about one. By the end of this notebook you will have:

1. Talked to a plain LLM and watched it fail at something it can't know
2. Run an agent that has a tool, and watched it decide to use that tool
3. **Written your own tool from scratch** and attached it to an agent
4. (Stretch) Given the agent a choice between two tools

**Run cells top to bottom with Shift+Enter.** Don't skip ahead — later cells assume earlier ones ran. If something errors, raise your hand — don't debug alone, the instructor/helper is walking around for exactly this.

> You do not need to understand every line of the setup cell below. That part is done for you. Your work starts at **Exercise 0**.

In [4]:
# === INSTRUCTOR SETUP -- pre-filled before the session, attendees just run this ===
#
# This cell configures WHICH model the agent talks to, via our internal vibe-gateway
# (LiteLLM proxy, OpenAI-compatible). Everything below this cell is what you'll
# actually be writing and reading.
#
# Use a FAST / CHEAP model alias here -- a room full of people hitting the gateway at
# once needs low latency and low cost, not the smartest model. Save a stronger model
# alias for the instructor's own live demo.

!pip install -q strands-agents strands-agents-tools openai

from strands import Agent, tool
from strands.models.openai import OpenAIModel

GATEWAY_API_KEY = "sk-cxfdPbHMfTjlZy4y0IWdDA"  # fill in: budget-capped key per attendee/session
GATEWAY_BASE_URL = "https://vibe-proxydev-westeurope-cdt.maersk.io"
GATEWAY_MODEL_ID = "claude-haiku-4-5"  # fill in: whichever CHEAP/FAST alias is registered on the gateway for the lab

lab_model = OpenAIModel(
    client_args={
        "api_key": GATEWAY_API_KEY,
        "base_url": GATEWAY_BASE_URL,
    },
    model_id=GATEWAY_MODEL_ID,
)

print("Setup complete. You're ready for Exercise 0.")


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Setup complete. You're ready for Exercise 0.


---
## Exercise 0 — Talk to a plain agent (no tools)

First, let's see what an LLM can and can't do on its own. Build an agent with **no tools** and ask it something ordinary, then something it has no way of actually knowing.

In [2]:
bare_agent = Agent(model=lab_model)

response = bare_agent("What is 12 times 8?")
# print(response)

12 times 8 is 96.

In [3]:
# Now ask it something it CANNOT know -- it has no tools, no access to your systems,
# and no memory beyond this conversation.
response = bare_agent("How many rows are in the `orders` table right now?")
# print(response)

I don't have access to any database or information about your `orders` table. I can't see your data or system.

To find out how many rows are in your `orders` table, you would need to run a SQL query like:

```sql
SELECT COUNT(*) FROM orders;
```

If you'd like help with this query or have questions about how to check table row counts, feel free to ask!

**Notice:** it either guesses, refuses, or makes something up ("hallucinates"). This isn't a bug
-- it genuinely has no way to know. That's exactly the gap tools exist to fill. Next exercise
gives it one.

---
## Exercise 1 — Give the agent a tool (already written)

Below is a tool: a normal Python function with a `@tool` decorator and a docstring. The
docstring is not a comment for humans -- it's what the **model reads** to decide whether and
when to call this function.

Read the function, then run both cells and watch what happens differently this time.

In [40]:
@tool
def check_table_stats(table_name: str) -> dict:
    """Look up row count, null percentage, and last-updated time for a table.

    Use this whenever the user asks about the health, freshness, or size of a
    specific table.

    Args:
        table_name: The name of the table to check, e.g. "orders".
    """
    # Fake data for this lab -- in a real agent this would query your warehouse.
    fake_db = {
        "orders": {"row_count": 1_204_331, "null_pct": 0.4, "last_updated": "2026-07-13 03:00 UTC"},
        "customers": {"row_count": 88_902, "null_pct": 2.1, "last_updated": "2026-07-12 22:15 UTC"},
    }
    return fake_db.get(table_name, {"error": f"no such table: {table_name}"})


tool_agent = Agent(
    model=lab_model,
    tools=[check_table_stats],
    system_prompt="You are a data reliability assistant. Use your tools to check real data before answering.",
)

response = tool_agent("How many rows are in the orders table, and when was it last updated?")
# print(response)


Tool #1: check_table_stats
The **orders** table has:
- **Row count:** 1,204,331 rows
- **Last updated:** July 13, 2026 at 03:00 UTC

The table is in good health with only 0.4% null values.

Look at the output above the final answer -- you should see the agent calling
`check_table_stats(table_name='orders')` before it responds. That's the loop from the intro:
**think (I need data) -> act (call the tool) -> observe (read the dict back) -> answer.**

Try changing the question below to ask about the `customers` table, or a table that doesn't
exist, and see how it handles each case.

In [42]:
# Try it yourself -- edit the question and re-run.
response = tool_agent("Is the customers table looking healthy?")
print(response)

Yes, the **customers** table is looking healthy! Here are the stats:

- **Row count:** 88,902 rows
- **Null percentage:** 2.1%
- **Last updated:** July 12, 2026 at 22:15 UTC

The table has a reasonable number of rows and the null percentage is relatively low at 2.1%, which is acceptable for most data quality standards. The data is fairly current, having been last updated about a day ago.Yes, the **customers** table is looking healthy! Here are the stats:

- **Row count:** 88,902 rows
- **Null percentage:** 2.1%
- **Last updated:** July 12, 2026 at 22:15 UTC

The table has a reasonable number of rows and the null percentage is relatively low at 2.1%, which is acceptable for most data quality standards. The data is fairly current, having been last updated about a day ago.



In [43]:
print(tool_agent.messages)

[{'role': 'user', 'content': [{'text': 'How many rows are in the orders table, and when was it last updated?'}], 'tracking_id': '7d72bc0d-8258-42bb-83e3-0a55a9571b28'}, {'role': 'assistant', 'content': [{'toolUse': {'toolUseId': 'tooluse_xbJleLM3cj1UOm1kVZ8cvl', 'name': 'check_table_stats', 'input': {'table_name': 'orders'}}}], 'metadata': {'usage': {'inputTokens': 655, 'outputTokens': 58, 'totalTokens': 713}, 'metrics': {'latencyMs': 0, 'timeToFirstByteMs': 2588}}, 'tracking_id': '51a451c2-210c-4479-9257-7d08b9f9c7fa'}, {'role': 'user', 'content': [{'toolResult': {'toolUseId': 'tooluse_xbJleLM3cj1UOm1kVZ8cvl', 'status': 'success', 'content': [{'text': '{"row_count": 1204331, "null_pct": 0.4, "last_updated": "2026-07-13 03:00 UTC"}'}]}}], 'tracking_id': 'b58d5b08-d0d1-45df-bf4d-5dca396b2deb'}, {'role': 'assistant', 'content': [{'text': 'The **orders** table has:\n- **Row count:** 1,204,331 rows\n- **Last updated:** July 13, 2026 at 03:00 UTC\n\nThe table is in good health with only 0.4

---
## Exercise 2 — Write your own tool (the main event)

Your turn. Below is a tool **stub** with a docstring already written and a `TODO` where the
logic should go. The tool checks whether a column in a fake table has any null values above a
threshold -- a small, realistic data-quality check.

You only need to fill in the body -- a few lines of plain Python, no new syntax.

In [45]:
# Pretend this is a row-level sample pulled from a table -- a list of dicts.
sample_rows = [
    {"order_id": 1, "customer_email": "a@example.com", "amount": 42.50},
    {"order_id": 2, "customer_email": None, "amount": 19.99},
    {"order_id": 3, "customer_email": "c@example.com", "amount": None},
    {"order_id": 4, "customer_email": None, "amount": 5.00},
]


@tool
def check_null_rate(column_name: str) -> dict:
    """Check what percentage of values in a column of the sample orders data are null.

    Use this when the user asks about missing values, data quality, or null rates
    for a specific column.

    Args:
        column_name: The column to check, e.g. "customer_email" or "amount".
    """
    # TODO: replace the line below.
    #
    # 1. Count how many rows in `sample_rows` have `row[column_name]` equal to None.
    # 2. Divide by the total number of rows to get a null rate (0.0 to 1.0).
    # 3. Return a dict like: {"column": column_name, "null_rate": <your number>}
    #
    # Hint: you can loop over sample_rows with a for-loop and a counter, or use
    # sum(1 for row in sample_rows if row[column_name] is None).

    null_count = sum(1 for row in sample_rows if row.get(column_name) is None)
    null_rate = null_count / len(sample_rows)
    return {"column": column_name, "null_rate": null_rate}

<details>
<summary>Stuck? Click here for the solution (try for a few minutes first!)</summary>

```python
@tool
def check_null_rate(column_name: str) -> dict:
    """Check what percentage of values in a column of the sample orders data are null.

    Use this when the user asks about missing values, data quality, or null rates
    for a specific column.

    Args:
        column_name: The column to check, e.g. "customer_email" or "amount".
    """
    null_count = sum(1 for row in sample_rows if row.get(column_name) is None)
    null_rate = null_count / len(sample_rows)
    return {"column": column_name, "null_rate": null_rate}
```
</details>

In [ ]:
# Once your tool is filled in, run this cell to re-run the cell above first (so the
# fixed version is loaded), then attach it to a fresh agent and ask a question that
# needs it.

my_agent = Agent(
    model=lab_model,
    tools=[check_null_rate],
    system_prompt="You are a data quality assistant. Use your tools to check real data before answering.",
)

response = my_agent("What percentage of customer email values are missing?")
# print(response)


Tool #1: check_null_rate
**50% of customer email values are missing** in the sample orders data.

This is a significant data quality issue - half of the records have no email address recorded. You may want to investigate:
- Why so many records lack email data
- Whether emails exist in another source or column
- If there are specific types of orders more likely to be missing email information
- Whether you need to collect this data or implement validation to prevent future gaps**50% of customer email values are missing** in the sample orders data.

This is a significant data quality issue - half of the records have no email address recorded. You may want to investigate:
- Why so many records lack email data
- Whether emails exist in another source or column
- If there are specific types of orders more likely to be missing email information
- Whether you need to collect this data or implement validation to prevent future gaps



### 2b — See why the docstring matters

Now let's break it on purpose. Copy your working tool below, but rewrite the docstring to be
vague and short (e.g. just `"""Does a thing."""`), **and rename the function itself to
something equally vague** (e.g. `tool_a`), then ask the same question. Does the agent still
call it?

**Why rename the function too:** the model doesn't only see the docstring -- it also sees the
function's *name* in the tool definition it's given. `check_null_rate_vague` still screams "I
check null rates" no matter what the docstring says, so the agent can call it correctly for the
wrong reason. To really test whether the docstring matters, every other signal (name included)
needs to be uninformative too. This is the single most useful debugging habit for tool-using
agents: **if the agent won't use your tool, suspect the description (and the name) before the
code.**

In [ ]:
@tool
def tool_a(column_name: str) -> dict:
    """Does a thing."""
    null_count = sum(1 for row in sample_rows if row.get(column_name) is None)
    null_rate = null_count / len(sample_rows)
    return {"column": column_name, "null_rate": null_rate}


vague_agent = Agent(model=lab_model, tools=[tool_a])
response = vague_agent("What percentage of customer_email values are missing?")
print(response)

---
## Exercise 3 (Stretch) — Give the agent a choice

Finished early? Attach **both** `check_table_stats` (from Exercise 1) and your
`check_null_rate` tool to the same agent, then ask a question that only one of them can answer,
and a question that touches both. Watch how the agent picks.

In [21]:
multi_tool_agent = Agent(
    model=lab_model,
    tools=[check_table_stats, check_null_rate],
    system_prompt="You are a data reliability and quality assistant. Use your tools to check real data before answering.",
)

response = multi_tool_agent(
    "Check the orders table's health, and also tell me what fraction of customer_email values are null."
)
# print(response)


Tool #1: check_table_stats

Tool #2: check_null_rate
Here's the health report for the **orders** table:

**Table Statistics:**
- **Row Count:** 1,204,331 rows
- **Overall Null Percentage:** 0.4%
- **Last Updated:** 2026-07-13 03:00 UTC

**Customer Email Column:**
- **Null Rate:** 50% of customer_email values are null

The orders table is reasonably fresh and large, but there's a significant data quality issue: **half of the customer_email values are missing**. This is notably higher than the table's overall null percentage of 0.4%, suggesting the customer_email column has substantial gaps that may impact downstream processes like customer communication or analysis.

---
## Wrap-up

You built an agent, watched the think -> act -> observe loop happen, and wrote your own tool.
That's the core mechanic behind almost every agent you'll hear about.

**Where this goes next** (not covered today -- look these up when you need them):

- **RAG** -- give the agent your own documents to search instead of hardcoded fake data
- **Multi-agent** -- have agents call other agents for sub-tasks
- **MCP** -- a standard way to plug in tools other people have already built
- **Real tools** -- swap the fake `sample_rows` / `fake_db` for an actual database query

**Homework idea:** take `check_null_rate` and point it at a real CSV or table you work with
instead of `sample_rows`, and try asking the agent a data-quality question you'd normally answer
by hand.

**Resources:** https://strandsagents.com/ for docs and more example tools.

---
## How would you actually deploy this?

Everything today ran inside a notebook cell that starts fresh each time. A real agent usually
lives inside a small **long-running service**, not a notebook. Nothing about the agent logic
changes -- same `Agent`, same `@tool` functions, same docstrings -- only *where it runs* and *how
it's called*:

- **Wrap it behind an API endpoint.** A small FastAPI/Flask service with one route (e.g.
  `POST /ask`) that does `agent(request.question)` and returns the response -- same three lines
  you wrote today, just called by a web request instead of a notebook cell.
- **Or trigger it on a schedule/event** instead of a live request -- a cron job or queue
  consumer that runs `agent(...)` on new data (e.g. "check every table every morning").
- **Secrets move out of the notebook.** Today's `api_key` was hardcoded in the setup cell for
  convenience. A deployed service reads it from an environment variable or a secrets manager --
  never committed to code.
- **Tools become real.** Swap `fake_db` / `sample_rows` for actual database queries or API calls
  -- the `@tool` function signature and docstring don't need to change at all.
- **No memory between calls, by default.** Each `agent(...)` call today started with no memory
  of the last one. A deployed service that needs multi-turn conversations has to store and
  replay conversation history itself (a session store, keyed by user/request).
- **You still go through the gateway.** Whether it's a notebook or a production service, model
  calls route through the same gateway (auth, budget caps, model routing) -- that's *why* it
  exists: so a deployed agent isn't holding a raw provider key either.

This is genuinely a "next rung," not something to build today -- infra, scaling, and
observability for agents are their own topic. The point to take away: **deploying an agent is
deploying a normal service that happens to call an LLM** -- there's no special "agent hosting"
magic beyond what you already know for shipping any other Python service.

### Try it yourself — run this agent behind a real HTTP endpoint

Let's make the bonus section above concrete. We'll take the exact agent from Exercise 1 — same
`check_table_stats` tool, same system prompt — and run it inside a small FastAPI service instead
of a notebook cell. The service runs as its own background process (the same trick Phase 2's MCP
server uses, if you've done that lab too), and once it's up you'll call it with a plain HTTP
request — exactly like calling any other web API. No notebook, no `Agent(...)` object, no
`@tool` decorator visible from the caller's side at all.

In [26]:
!pip install -q fastapi uvicorn

In [27]:
import os

# Secrets move out of the notebook and into the deployed process's environment --
# this is how the service file below reads them, instead of hardcoding them again.
os.environ["GATEWAY_API_KEY"] = GATEWAY_API_KEY
os.environ["GATEWAY_BASE_URL"] = GATEWAY_BASE_URL
os.environ["GATEWAY_MODEL_ID"] = GATEWAY_MODEL_ID

In [28]:
%%writefile agent_service.py
import os

from fastapi import FastAPI
from pydantic import BaseModel
from strands import Agent, tool
from strands.models.openai import OpenAIModel

app = FastAPI()

model = OpenAIModel(
    client_args={
        "api_key": os.environ["GATEWAY_API_KEY"],
        "base_url": os.environ["GATEWAY_BASE_URL"],
    },
    model_id=os.environ["GATEWAY_MODEL_ID"],
)


@tool
def check_table_stats(table_name: str) -> dict:
    """Look up row count, null percentage, and last-updated time for a table.

    Use this whenever the user asks about the health, freshness, or size of a
    specific table.

    Args:
        table_name: The name of the table to check, e.g. "orders".
    """
    fake_db = {
        "orders": {"row_count": 1_204_331, "null_pct": 0.4, "last_updated": "2026-07-13 03:00 UTC"},
        "customers": {"row_count": 88_902, "null_pct": 2.1, "last_updated": "2026-07-12 22:15 UTC"},
    }
    return fake_db.get(table_name, {"error": f"no such table: {table_name}"})


agent = Agent(
    model=model,
    tools=[check_table_stats],
    system_prompt="You are a data reliability assistant. Use your tools to check real data before answering.",
)


class AskRequest(BaseModel):
    question: str


@app.post("/ask")
def ask(request: AskRequest):
    response = agent(request.question)
    return {"answer": str(response)}

Overwriting agent_service.py


In [29]:
import subprocess
import time

import requests

# Launch the service as its own background process -- same idea as the MCP server
# subprocess, just speaking HTTP instead of stdio.
service_proc = subprocess.Popen(["uvicorn", "agent_service:app", "--port", "8000"])

# Give it a moment to finish starting up before we call it.
for _ in range(15):
    try:
        requests.get("http://127.0.0.1:8000/docs", timeout=1)
        break
    except requests.exceptions.ConnectionError:
        time.sleep(1)

print("Service is up at http://127.0.0.1:8000")

INFO:     Started server process [37849]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:53669 - "GET /docs HTTP/1.1" 200 OK
Service is up at http://127.0.0.1:8000


INFO:     127.0.0.1:53677 - "GET / HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:53677 - "GET /favicon.ico HTTP/1.1" 404 Not Found


In [30]:
# Call it exactly like you'd call any other web API -- a URL and a JSON body.
# No import of `strands`, no `Agent`, no visibility into tools from this side at all.
resp = requests.post(
    "http://127.0.0.1:8000/ask",
    json={"question": "How many rows are in the orders table, and when was it last updated?"},
)
print(resp.json()["answer"])


Tool #1: check_table_stats
The **orders** table contains **1,204,331 rows** and was last updated on **July 13, 2026 at 03:00 UTC**.

Additionally, the table has a null percentage of 0.4%, indicating good data quality with very few null values.INFO:     127.0.0.1:53686 - "POST /ask HTTP/1.1" 200 OK
The **orders** table contains **1,204,331 rows** and was last updated on **July 13, 2026 at 03:00 UTC**.

Additionally, the table has a null percentage of 0.4%, indicating good data quality with very few null values.



Based on the stats I just checked, the **orders table is in very good health**:

1. **Row Count**: 1,204,331 rows - a substantial dataset, indicating the table is actively populated and being used.

2. **Data Quality**: 0.4% null percentage is excellent. This means 99.6% of the data is complete, which indicates high data quality with minimal missing values.

3. **Freshness**: Last updated on July 13, 2026 at 03:00 UTC - this shows the table is being actively maintained and kept current.

**Overall Assessment**: The orders table appears to be healthy and reliable. The combination of a good volume of data, minimal null values, and recent updates suggests it's well-maintained and suitable for analysis and reporting.INFO:     127.0.0.1:53787 - "POST /ask HTTP/1.1" 200 OK


**Notice:** the response came back from a URL, not a Python object. The caller has no idea
there's an LLM, a tool, or a think → act → observe loop behind `/ask` — it just sees a normal
JSON API. That's the entire point of "deploying an agent": the agent logic doesn't change, only
what sits in front of it.

Run the cell below when you're done experimenting, to stop the background service (Colab will
also kill it automatically when the runtime disconnects).

In [31]:
service_proc.terminate()
service_proc.wait()
print("Service stopped.")

Service stopped.


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [37849]


---
## Exercise 4 — Give the agent memory

Every agent you built so far was called once per cell, with a fresh question. But `Agent`
objects aren't stateless — call the **same object** twice, and it remembers what was said
before, just like a chat conversation.

Below, we ask a follow-up question that only makes sense if the agent remembers the first
answer.

In [34]:
memory_agent = Agent(
    model=lab_model,
    tools=[check_table_stats],
    system_prompt="You are a data reliability assistant. Use your tools to check real data before answering.",
)

response = memory_agent("How many rows are in the orders table?")
# print(response)


Tool #1: check_table_stats
The **orders** table contains **1,204,331 rows**. 

The table was last updated on **2026-07-13 at 03:00 UTC** and has a null percentage of 0.4%, which indicates good data quality.

In [36]:
# Notice: we don't repeat "orders table" here -- "that" only resolves because
# memory_agent remembers the previous turn.
response = memory_agent("Is that a lot compared to the customers table?")
# print(response)

Yes, the **orders** table is significantly larger than the **customers** table:

- **Orders**: 1,204,331 rows
- **Customers**: 88,902 rows

The orders table has about **13.5x more rows** than the customers table, which makes sense since each customer can have multiple orders. This ratio suggests an average of roughly 13-14 orders per customer in your dataset.

Note that the customers table also has a slightly higher null percentage (2.1% vs 0.4%), and it was last updated slightly earlier (2026-07-12 at 22:15 UTC vs 2026-07-13 at 03:00 UTC for orders).

In [39]:
print(memory_agent.messages)

[{'role': 'user', 'content': [{'text': 'How many rows are in the orders table?'}], 'tracking_id': '37f6ce5f-684e-482a-bd33-6ddb8934a20a'}, {'role': 'assistant', 'content': [{'toolUse': {'toolUseId': 'tooluse_dvVBWzd09jKU6mk2qnu3gK', 'name': 'check_table_stats', 'input': {'table_name': 'orders'}}}], 'metadata': {'usage': {'inputTokens': 648, 'outputTokens': 58, 'totalTokens': 706}, 'metrics': {'latencyMs': 0, 'timeToFirstByteMs': 3106}}, 'tracking_id': 'dc7ce846-0c5c-4fbc-b7b4-6c4ce385e30b'}, {'role': 'user', 'content': [{'toolResult': {'toolUseId': 'tooluse_dvVBWzd09jKU6mk2qnu3gK', 'status': 'success', 'content': [{'text': '{"row_count": 1204331, "null_pct": 0.4, "last_updated": "2026-07-13 03:00 UTC"}'}]}}], 'tracking_id': 'bbf5b67d-0faf-4fb5-99e7-cf35590832c9'}, {'role': 'assistant', 'content': [{'text': 'The **orders** table contains **1,204,331 rows**. \n\nThe table was last updated on **2026-07-13 at 03:00 UTC** and has a null percentage of 0.4%, which indicates good data quality.

In [41]:
# Now compare against a BRAND NEW agent that never saw the first question.
fresh_agent = Agent(model=lab_model, tools=[check_table_stats])
response = fresh_agent("Is that a lot compared to the customers table?")
print(response)

I'd be happy to help you compare table sizes! However, I need to know which table you're referring to. Could you please specify:

1. Which table you want me to check first (the one you're asking about)?
2. Confirm you want me to compare it to the `customers` table?

Once you provide the name of the first table, I can check the stats for both and compare them for you.I'd be happy to help you compare table sizes! However, I need to know which table you're referring to. Could you please specify:

1. Which table you want me to check first (the one you're asking about)?
2. Confirm you want me to compare it to the `customers` table?

Once you provide the name of the first table, I can check the stats for both and compare them for you.



**Notice:** `fresh_agent` has no idea what "that" refers to — it either asks for clarification
or guesses, because it has no memory of the earlier conversation. Same model, same tool,
same question — the only difference is which object you called.

This is exactly what the deployment section earlier warned about: "no memory between calls,
by default." A notebook `Agent` object keeps memory for as long as it stays in memory (pun
intended) — a deployed service has to deliberately store and replay conversation history
per user/session, or every request starts from zero.

---
## Exercise 5 — Human-in-the-loop (HITL): pause before anything destructive

Every tool so far only *read* data. What if a tool could **change** something — like deleting
old rows? You don't want the agent doing that unsupervised just because it decided to.

The fix is simple: put the pause **inside the tool itself**, before the destructive part runs.
The tool below asks a human to type `y`/`n` before it "deletes" anything. Run it, and try
answering both ways.

In [ ]:
@tool
def delete_stale_rows(table_name: str) -> dict:
    """Delete rows older than 1 year from a table. THIS IS DESTRUCTIVE.

    Use this only when the user explicitly asks to delete or purge old/stale
    data from a specific table.

    Args:
        table_name: The table to delete stale rows from, e.g. "orders".
    """
    # This is the HITL checkpoint: the tool pauses and waits for a human
    # before it does anything irreversible. No approval, no deletion.
    answer = input(f"Agent wants to delete stale rows from '{table_name}'. Proceed? (y/n): ")
    if answer.strip().lower() != "y":
        return {"status": "cancelled", "reason": "not approved by human"}

    # Pretend this actually deletes rows -- fake for the lab.
    return {"status": "deleted", "table": table_name, "rows_deleted": 4213}


hitl_agent = Agent(
    model=lab_model,
    tools=[delete_stale_rows],
    system_prompt="You are a data maintenance assistant. Use your tools when asked to clean up data.",
)

response = hitl_agent("Please delete stale rows from the orders table.")
# print(response)


Tool #1: delete_stale_rows
The deletion request was not approved. This is a safety measure because deleting data is destructive and permanent. 

To proceed with deleting stale rows (older than 1 year) from the orders table, please confirm that you want to proceed with this action. This will permanently remove all records from the orders table that are older than 1 year.The deletion request was not approved. This is a safety measure because deleting data is destructive and permanent. 

To proceed with deleting stale rows (older than 1 year) from the orders table, please confirm that you want to proceed with this action. This will permanently remove all records from the orders table that are older than 1 year.



**Try both paths:** re-run the cell above once and type `y`, then run it again and type `n`.
Notice the agent handles the "cancelled" result gracefully — it just reports back that the
human said no, instead of failing.

**Notice:** the model never got a choice about *whether* to ask for approval — the tool forces
the pause every single time it's called, before the "deletion" line runs. That's the core idea
of human-in-the-loop: **don't rely on the agent to decide when to ask permission — build the
checkpoint into the tool so it's unskippable.**

A real production version wouldn't use `input()` — it would write a pending request to a
queue/UI and wait for a human to click "approve" there — but the mechanic is identical: pause
before the irreversible action, not before the decision to call the tool.

---
## Exercise 6 — A simple multi-agent system

So far, one agent has juggled all its own tools. But as an agent picks up more tools, one giant
system prompt trying to describe everything gets messy — and some questions genuinely need a
different kind of specialist altogether, not just a different tool.

**The use case:** a tutoring system that has to handle three different kinds of student
questions:

1. A **physics-only** question (no numbers) — e.g. "why does the sky look blue?"
2. A **maths-only** question (no physics concept) — e.g. "what's 15% of 240?"
3. A **combined** question that needs physics *and* maths — e.g. a projectile motion problem,
   where you need the right formula (physics) before you can compute a number (maths).

The **orchestrator**'s job is to look at the question, decide which of these three cases it is,
and run the right agent(s) — calling physics then maths in sequence only when both are actually
needed, and calling just one specialist directly otherwise.

We'll build:
- a **router agent** — reads the question and decides: `physics`, `maths`, or `both`
- a **physics specialist** — explains the concept and states the formula, no arithmetic
- a **maths specialist** — computes a result, either from a formula it's handed or on its own
- a **curator agent** — only used for the combined case, to merge concept + result into one
  answer

In [ ]:
# The router doesn't need any tools -- its only job is to read a question and say
# which specialist(s) should handle it.
router_agent = Agent(
    model=lab_model,
    system_prompt="You are a router for a tutoring system. Given a student's question, "
    "decide which specialist(s) are needed: 'physics' (concept questions, no numbers to "
    "compute), 'maths' (pure calculation, no physics concept involved), or 'both' (needs a "
    "physics formula AND a calculation, e.g. projectile motion). "
    "Respond with ONLY one word: physics, maths, or both -- no other text.",
)

# Specialist 1: explains the concept AND states the formula -- but never plugs in numbers.
physics_agent = Agent(
    model=lab_model,
    system_prompt="You are a physics tutor. Given a student's question, respond in AT MOST "
    "3 short sentences, plain text, no markdown headers, no LaTeX:\n"
    "1. One sentence on the physics concept (why it behaves this way).\n"
    "2. One line stating the formula needed, e.g. 'Formula: distance = speed x time'.\n"
    "Do NOT plug in numbers or calculate a final answer -- that's the maths specialist's job.",
)

# Specialist 2: computes a result -- either from a formula it's handed (combined case),
# or straight from the question itself (maths-only case).
maths_agent = Agent(
    model=lab_model,
    system_prompt="You are a maths tutor. If you're given a formula, plug the numbers from "
    "the question into it and compute the result. If no formula is given, just work out "
    "the calculation directly. Respond in AT MOST 3 short lines, plain text, no markdown "
    "headers, no LaTeX -- just the steps and the final number. Do NOT explain any physics.",
)

# The curator is only needed for the "both" case: merging concept + computed result
# into one clean answer instead of two stapled-together notes.
curator_agent = Agent(
    model=lab_model,
    system_prompt="You are a tutoring orchestrator. You'll be given a student's original "
    "question, a physics concept + formula, and a computed result. Combine them into ONE "
    "short final answer for the student: 1 sentence concept, 1 sentence with the number. "
    "Plain text, no markdown headers, no LaTeX, no repeated information.",
)

In [ ]:
# The orchestrator: route first, then run only the agent(s) that question actually needs.
#
# Note: calling an Agent (e.g. physics_agent(question)) already streams/prints its answer
# to the notebook automatically -- we don't need to print it ourselves. We only print a
# short header before each call so it's clear WHICH agent is running WHEN.
def orchestrate(question: str) -> str:
    decision = str(router_agent(question)).strip().lower()
    print(f"[router decided: {decision}]")

    if "both" in decision:
        print("\n===== STEP 1: physics_agent -- concept + formula =====")
        physics_notes = str(physics_agent(question))

        print("\n===== STEP 2: maths_agent -- plug in the numbers =====")
        maths_result = str(maths_agent(
            f"Question: {question}\n\nFormula from the physics specialist:\n{physics_notes}\n\n"
            "Using this formula and the numbers in the question, calculate the answer."
        ))

        print("\n===== STEP 3: curator_agent -- final combined answer =====")
        return str(curator_agent(
            f"Student's question: {question}\n\nPhysics concept + formula:\n{physics_notes}\n\n"
            f"Computed result:\n{maths_result}\n\nWrite ONE short final answer."
        ))

    if "physics" in decision:
        print("\n===== STEP 1: physics_agent only (no maths needed) =====")
        return str(physics_agent(question))

    print("\n===== STEP 1: maths_agent only (no physics concept needed) =====")
    return str(maths_agent(question))


# 1. Physics-only -- no numbers to compute.
orchestrate("Why does the sky look blue during the day?")

In [ ]:
# 2. Maths-only -- pure calculation, no physics concept involved.
orchestrate("What is 15% of 240?")

In [ ]:
# 3. Combined -- needs the physics formula BEFORE the maths calculation can happen.
result = orchestrate(
    "A ball is thrown horizontally at 10 m/s off a 20m cliff. How far does it travel "
    "before hitting the ground, and why does its horizontal speed stay constant?"
)

**Compare the three runs above:**

- **"Why does the sky look blue?"** → router said `physics` → only `physics_agent` ran. No
  wasted call to `maths_agent` for a question with nothing to calculate.
- **"What is 15% of 240?"** → router said `maths` → only `maths_agent` ran, straight from the
  question -- no formula handed to it, because there's no physics concept here at all.
- **The projectile motion question** → router said `both` → `physics_agent` ran first to
  produce a formula, `maths_agent` ran second using that exact formula, and `curator_agent` ran
  last to merge them into one answer.

That's the actual shape of a useful multi-agent system: **the orchestrator's first job is
deciding how much work is even needed**, and only the "both" path pays for three agent calls
instead of one. `curator_agent` only shows up when there are two outputs that genuinely need
merging -- for the single-specialist cases, its answer already reads as one clean response, so
adding a curation step there would just be extra cost for no benefit.

Try it yourself: write a maths-only question that's easy to get wrong if physics sneaks in
(e.g. "what's the square root of 144?"), and a physics-only question with a number in it that
still doesn't require calculation (e.g. "does a 5kg ball fall faster than a 1kg ball?") --
watch the router still classify both correctly.